## Scraper — Historico de Times (Basketball Reference)

Fonte: https://www.basketball-reference.com/teams/
Saida: basket_dbt/seeds/team.csv

In [1]:
from __future__ import annotations
import logging, time
from io import StringIO
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup, Comment
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service

In [2]:
logger = logging.getLogger('bbr_teams')
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', '%Y-%m-%d %H:%M:%S'))
    logger.addHandler(h)
logger.setLevel(logging.INFO)

In [3]:
CHROMEDRIVER_PATH = '/snap/bin/chromium.chromedriver'
CHROME_BINARY     = '/usr/bin/chromium-browser'

# Renomeia colunas invalidas para SQL
COLUMN_RENAME = {
    'W/L%': 'wl_pct',
}

def get_rendered_html(url: str, wait_seconds: int = 8) -> str:
    logger.info('Abrindo: %s', url)
    opts = ChromeOptions()
    opts.binary_location = CHROME_BINARY
    opts.add_argument('--headless=new')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/127.0.0.0 Safari/537.36')
    driver = webdriver.Chrome(service=Service(CHROMEDRIVER_PATH), options=opts)
    try:
        driver.get(url)
        time.sleep(wait_seconds)
        html = driver.page_source
        logger.info('HTML obtido (%d chars).', len(html))
        return html
    finally:
        driver.quit()

def uncomment_tables(raw_html: str) -> BeautifulSoup:
    soup = BeautifulSoup(raw_html, 'lxml')
    comments = soup.find_all(string=lambda t: isinstance(t, Comment))
    for c in comments:
        c.replace_with(BeautifulSoup(c, 'lxml'))
    logger.info('%d comentario(s) descomentados.', len(comments))
    return soup

def extract_table(soup: BeautifulSoup, table_id: str) -> pd.DataFrame:
    table = soup.select_one(f'table#{table_id}')
    if not table:
        raise RuntimeError(f'Tabela {table_id!r} nao encontrada.')
    df = pd.read_html(StringIO(str(table)))[0]
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [' '.join(map(str, col)).strip() for col in df.columns]
    logger.info('Tabela %r: %d linhas, %d colunas.', table_id, *df.shape)
    return df

In [4]:
URL      = 'https://www.basketball-reference.com/teams/'
TABLE_ID = 'teams_active'
OUT_PATH = Path('../../basket_dbt/seeds/team.csv')

html   = get_rendered_html(url=URL)
soup   = uncomment_tables(html)
df_raw = extract_table(soup, TABLE_ID)
print('Colunas:', df_raw.columns.tolist())

2026-04-03 14:39:27 | INFO | Abrindo: https://www.basketball-reference.com/teams/


2026-04-03 14:39:37 | INFO | HTML obtido (343699 chars).


2026-04-03 14:39:37 | INFO | 126 comentario(s) descomentados.


2026-04-03 14:39:37 | INFO | Tabela 'teams_active': 87 linhas, 13 colunas.


Colunas: ['Franchise', 'Lg', 'From', 'To', 'Yrs', 'G', 'W', 'L', 'W/L%', 'Plyfs', 'Div', 'Conf', 'Champ']


In [5]:
df = df_raw.rename(columns=COLUMN_RENAME).copy()
logger.info('Times/franciquias: %d', len(df))
print(df.head(5).to_string())

2026-04-03 14:39:37 | INFO | Times/franciquias: 87


               Franchise   Lg     From       To Yrs     G     W     L wl_pct Plyfs Div Conf Champ
0          Atlanta Hawks  NBA  1949-50  2025-26  77  6097  3011  3085   .494    49  12    0     1
1          Atlanta Hawks  NBA  1968-69  2025-26  58  4678  2313  2365   .494    36   6    0     0
2        St. Louis Hawks  NBA  1955-56  1967-68  13  1005   553   452   .550    12   6    0     1
3        Milwaukee Hawks  NBA  1951-52  1954-55   4   282    91   190   .324     0   0    0     0
4  Tri-Cities Blackhawks  NBA  1949-50  1950-51   2   132    54    78   .409     1   0    0     0


In [6]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False, encoding='utf-8')
logger.info('Salvo em: %s (%d linhas)', OUT_PATH.resolve(), len(df))

2026-04-03 14:39:37 | INFO | Salvo em: /home/henri/Basketanalysis/basket_dbt/seeds/team.csv (87 linhas)
